In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

IN_JSON = "data/lod3_meshes.json"               # LOD3 (10_lod3)
JSON_FILE = "data/lod3_volumenes_meshes.json"   # Read by 14_materiales
TERRAIN_JSON = "data/colegio_imperial_fotos_meshes.json"   # Terrain of 13_cubiertas (ground under the walls)
ORIGIN = (440000.0, 4473700.0, 640.0)

# Tables (one row per level of each piece, and one row per group of openings), edited by hand:
VOLUMES_CSV = "data/volumenes/volumenes.csv"
OPENINGS_CSV = "data/volumenes/huecos.csv"
CHAPELS_CSV = "data/volumenes/capillas.csv"   # Buttresses, skylight and windows of the chapels (Celda 7)
GABLE_CSV = "data/volumenes/hastial.csv"      # West gable of the nave, between the towers (Celda 7)

# Frames of the pieces (EPSG:25830). Each piece is a ring of 8 vertices per level, in a local frame:
# u along "rotation" (degrees from east, counterclockwise), v 90 degrees to the left of u.
# Dome: octagon fitted to the building points between 678.5 and 691 m (center and rotation that make the
# apothem most constant in each 1 m slice); the faces point at 34 + 45 k degrees.
# Towers: center fitted to the cloud between 682 and 685 m; u along the Toledo facade (81.5 degrees), v toward the
# street. The tower is moved along v so that its front lies on the facade line of the Catastro, where the cloud
# also puts it (within 0.3 m), and where the openings measured on the facade are: see NOTES.md.
TOLEDO_LINE = [(439988.6, 4473740.7), (439984.4, 4473757.7), (439978.7, 4473780.4), (439979.4, 4473792.5),
               (439980.8, 4473819.3), (439982.0, 4473827.4), (439984.2, 4473842.1), (439985.5, 4473850.4)]
FRAMES = {
    "cupula": {"center": (440028.24, 4473826.91), "rotation": 34.0},
    "linterna": {"center": (440028.24, 4473826.91), "rotation": 34.0},
    "torre_s": {"center": (439985.12, 4473823.02), "rotation": 81.5, "front_on_facade": True},
    "torre_n": {"center": (439988.81, 4473846.02), "rotation": 81.5, "front_on_facade": True},
    # Octagonal turret on the north chapels: center of the building points above 667 m; faces along the nave.
    "torrecilla": {"center": (440008.02, 4473849.05), "rotation": -8.5},
    # Nave of the Colegiata (as in MANUAL_BAYS): u along the axis toward the crossing, v to the north. Frame of the
    # elements of CHAPELS_CSV.
    "nave": {"center": (440005.0, 4473831.8), "rotation": -8.5},
}

# The LOD2 grid is cut flat at these heights under the footprint of the towers and the dome, which the new
# pieces replace:
# Dome: the flat roof of the crossing around the drum is at 672.85 m (part 8, cloud).
CLIP_Z = {"torre_s": 664.0, "torre_n": 664.0, "cupula": 672.85}
CLIP_MARGIN = 1.0   # Meters around the footprint of each piece

# LOD3 facade elements replaced by the new towers (they were drawn on the facade plane, not on the set-back
# belfry), and the old tower spires (moved with the towers):
DROP_ELEMENTS = ["torre_n_campanario", "torre_s_campanario", "torre_n_cornisa_media", "torre_s_cornisa_media",
                 "remate_torre_norte", "remate_torre_sur"]

# Spires and crosses on the new towers: tip heights from the cloud (as in 10_lod3); sizes [ESTIMADO].
TOWER_SPIRES = {"torre_n": 689.04, "torre_s": 689.12}
SPIRE_RADIUS = 0.35
CROSS_HEIGHT = 1.5
CROSS_ARM = 0.9
CROSS_BAR = 0.12
ARCH_SEGMENTS = 12
BAR_SPACING = 0.2   # Iron grilles: one vertical bar every 0.2 m [ESTIMADO]

# Roofs by bays ("crujías"): each LOD2 part of the ensemble is fitted with simple roofs: flat, one plane, or two
# pitches with a ridge along either side of the part; if none explains the cloud, the part is cut in two by the
# straight line that most lowers the error, and so on (up to ROOF_MAX_DEPTH times). Error = mean absolute residual
# of the building points, each one truncated at 2 m. A part whose final error stays above ROOF_KEEP keeps its grid.
GPKG = "data/catastro_manzana.gpkg"
CLOUD = "data/nube_manzana_2023.laz"
ROOF_OK = 0.15        # A piece is done when its error is below this (m)
ROOF_KEEP = 0.45      # Parts with a larger final error keep the grid
ROOF_MIN_AREA = 25.0  # Smallest piece (m2)
ROOF_MAX_DEPTH = 3    # Up to 2^3 = 8 pieces per part
ROOF_STEP = 0.25      # Ridge positions tried every this many meters
ROOF_MAX_POINTS = 20000
ROOF_SIMPLER = 0.03   # The simplest model within this error of the best one wins
ROOF_REPORT = "data/volumenes/cubiertas_ajuste.csv"
KEEP_GRID = ["0139402VK4703G_part6", "0139402VK4703G_part9", "0139402VK4703G_part12"]   # Towers and dome (replaced below)
FORCE_BAYS = ["0139402VK4703G_part11", "0139402VK4703G_part14"]   # Small parts: bays even if the error stays high
ROOF_CUT_GAIN = 0.9   # A cut is kept if it lowers the error to less than this fraction
# Eaves: on the instituto the tiles run out over the stone cornice (photo 01), up to its projection (0.6 m, [ESTIMADO]
# in the facade table). Only toward streets and courtyards (outside the footprints of the block).
EAVE = 0.6
EAVE_PARCELS = ["0139424VK4703G", "0139423VK4703G"]
# Bays drawn by hand (center x, y in local meters, length, width, angle in degrees), for parts that the automatic cuts
# do not solve. The rest of the part is cut automatically. These parts are always replaced (no ROOF_KEEP test).
# Nave of the Colegiata: rectangle along the axis of the church (-8.5 degrees, perpendicular to the Toledo facade),
# read on the hillshade of the cloud; the ridge, eaves and slopes are fitted. The side chapels keep an error of
# 0.4-0.9 m because one pitch does not draw the buttresses between them [simplification].
MANUAL_BAYS = {
    "0139402VK4703G_part5": [(5.0, 131.8, 31.0, 18.0, -8.5)],
    # Wings around the cloister (instituto): west (Toledo bay, along the courtyard side at 86.5 degrees), then north,
    # south and east between the courtyard (Catastro hole) and the outer walls. Each one takes what the previous ones
    # left. The east annex is cut automatically.
    "0139424VK4703G_part1": [(-12.6, 99.0, 44.0, 15.5, 86.5), (6.0, 111.5, 24.0, 9.0, -6.0),
                             (5.0, 83.5, 22.0, 12.0, -4.0), (19.0, 101.0, 24.0, 14.0, 86.0)],
}

In [ ]:
# Celda 2
import csv
import json
import math
import numpy as np
import laspy
import geopandas as gpd
import shapely
from shapely.geometry import LineString, Point
from shapely.geometry.polygon import orient
from shapely.ops import split

with open(VOLUMES_CSV, encoding="utf-8") as f:
    levels = list(csv.DictReader(f))
with open(OPENINGS_CSV, encoding="utf-8") as f:
    openings = list(csv.DictReader(f))
pieces = {}
for row in levels:
    pieces.setdefault(row["pieza"], []).append(row)
print(len(pieces), "pieces |", len(openings), "groups of openings")

In [ ]:
# Celda 3
# Frame of each piece: center, unit vectors u and v. Towers: moved along v so that the front face of the belfry
# (v = hv) lies on the facade line.
def frame_name(piece):
    return next(name for name in FRAMES if piece.startswith(name))

facade = LineString(TOLEDO_LINE)
frames = {}
for name, frame in FRAMES.items():
    angle = math.radians(frame["rotation"])
    u = (math.cos(angle), math.sin(angle))
    v = (-math.sin(angle), math.cos(angle))
    cx, cy = frame["center"]
    if frame.get("front_on_facade"):
        belfry_hv = float(pieces[name + "_campanario"][0]["hv"])
        distance = facade.distance(Point(cx, cy))
        shift = distance - belfry_hv   # Negative: the tower goes back from the street
        cx, cy = cx + v[0] * shift, cy + v[1] * shift
        print(name, "moved", round(shift, 2), "m along v (toward the street if positive)")
    frames[name] = {"center": (cx, cy), "u": u, "v": v}

def to_local(name, a, b, z):
    """(a, b) in the frame of the piece -> local Blender coordinates."""
    f = frames[name]
    x = f["center"][0] + a * f["u"][0] + b * f["v"][0]
    y = f["center"][1] + a * f["u"][1] + b * f["v"][1]
    return (x - ORIGIN[0], y - ORIGIN[1], z - ORIGIN[2])

def ring(hu, hv, chamfer):
    """8 corners of a rectangle of half sides hu x hv with the corners cut (chamfer = leg of the cut), counterclockwise
    from the end of face 0 (+u). Faces: 0 = +u, 2 = +v, 4 = -u, 6 = -v, odd = chamfers.
    "regular": regular octagon of apothem hu."""
    if chamfer == "regular":
        c = 2 * hu / (2 + math.sqrt(2))
    else:
        c = max(float(chamfer), 0.001)   # A tiny cut keeps 8 corners on square pieces
    return [(hu, -hv + c), (hu, hv - c), (hu - c, hv), (-hu + c, hv), (-hu, hv - c), (-hu, -hv + c), (-hu + c, -hv), (hu - c, -hv)]

In [ ]:
# Celda 4
# Cornices: the two rows of a cornice become one ring per point of a moulded profile (fascia, cyma, corona, cymatium:
# the same profile as the cornices of the facades, 10_lod3), from the size of the piece below to the full projection.
CORNICE_PROFILE = [(0, 0), (0.15, 0), (0.15, 0.12), (0.25, 0.2), (0.35, 0.3), (0.45, 0.36), (0.45, 0.42), (0.95, 0.45),
                   (1.0, 0.5), (1.0, 0.72), (0.92, 0.76), (0.92, 0.8), (0.97, 0.9), (1.0, 1.0), (0, 1.0)]

def profiled(name, rows):
    names = list(pieces)
    if "cornisa" not in name or len(rows) != 2 or names.index(name) == 0:
        return rows
    below = pieces[names[names.index(name) - 1]][-1]
    z1, z2 = float(rows[0]["z"]), float(rows[1]["z"])
    def mix(key, fn):
        a, b = below[key], rows[0][key]
        return "regular" if "regular" in (a, b) else str(float(a) + fn * (float(b) - float(a)))
    return [{**rows[0], "z": str(z1 + fh * (z2 - z1)), "hu": mix("hu", fn), "hv": mix("hv", fn), "chaflan": mix("chaflan", fn)}
            for fn, fh in CORNICE_PROFILE]

# Solid of each piece: one ring per row of the table, joined by quads, closed at the bottom and the top.
def piece_mesh(name, rows):
    rows = profiled(name, rows)
    frame = frame_name(name)
    vertices, faces = [], []
    for row in rows:
        for a, b in ring(float(row["hu"]), float(row["hv"]), row["chaflan"]):
            vertices.append(to_local(frame, a, b, float(row["z"])))
    for level in range(len(rows) - 1):
        for k in range(8):
            j = (k + 1) % 8
            faces.append((level * 8 + k, level * 8 + j, (level + 1) * 8 + j, (level + 1) * 8 + k))
    faces.append(tuple(reversed(range(8))))                         # Bottom
    faces.append(tuple((len(rows) - 1) * 8 + k for k in range(8)))  # Top
    return vertices, faces

# Openings and strips on the faces of a piece, at the height of their middle: a plate from the face outward.
def face_at(name, face, z):
    """Start and end of a face (a, b) at height z, by interpolating the table."""
    rows = pieces[name]
    zs = [float(r["z"]) for r in rows]
    k = max(i for i in range(len(rows)) if zs[i] <= z) if z >= zs[0] else 0
    k = min(k, len(rows) - 2)
    t = 0 if zs[k + 1] == zs[k] else (z - zs[k]) / (zs[k + 1] - zs[k])
    corners = []
    for row in (rows[k], rows[k + 1]):
        corners.append(ring(float(row["hu"]), float(row["hv"]), row["chaflan"]))
    a0 = [(1 - t) * p + t * q for p, q in zip(corners[0][face], corners[1][face])]
    a1 = [(1 - t) * p + t * q for p, q in zip(corners[0][(face + 1) % 8], corners[1][(face + 1) % 8])]
    return a0, a1

def plate(name, face, s1, s2, z1, z2, n1, n2, outline=None):
    """Box on a face: s = meters along the face from its middle, n = meters out of the face. With outline (list of
    (s, z)), a flat prism of that shape instead of a rectangle."""
    frame = frame_name(name)
    a0, a1 = face_at(name, face, (z1 + z2) / 2)
    length = math.dist(a0, a1)
    ta = ((a1[0] - a0[0]) / length, (a1[1] - a0[1]) / length)
    out = (ta[1], -ta[0])                                   # The ring goes counterclockwise: outward is on the right
    middle = ((a0[0] + a1[0]) / 2, (a0[1] + a1[1]) / 2)
    def point(s, n, z):
        return to_local(frame, middle[0] + ta[0] * s + out[0] * n, middle[1] + ta[1] * s + out[1] * n, z)
    if outline is None:
        outline = [(s1, z1), (s2, z1), (s2, z2), (s1, z2)]
    vertices = [point(s, n1, z) for s, z in outline] + [point(s, n2, z) for s, z in outline]
    m = len(outline)
    faces = [tuple(reversed(range(m))), tuple(range(m, 2 * m))]
    for k in range(m):
        j = (k + 1) % m
        faces.append((k, j, m + j, m + k))
    return vertices, faces, length

def opening_meshes(row):
    name = row["pieza"]
    faces_list = range(8) if row["caras"] == "todas" else [int(f) for f in row["caras"].split(";")]
    width, z1, z2 = float(row["ancho"]), float(row["z1"]), float(row["z2"])
    depth = float(row["vuelo"])
    meshes = []
    for face in faces_list:
        _, _, length = plate(name, face, 0, 1, z1, z2, 0, depth)
        half = width / 2
        if row["forma"] == "rect":
            meshes.append(plate(name, face, -half, half, z1, z2, 0, depth)[:2])
        elif row["forma"] == "cara":                       # The whole face
            meshes.append(plate(name, face, -length / 2, length / 2, z1, z2, 0, depth)[:2])
        elif row["forma"] == "esquinas":                   # Two strips at the ends of the face
            meshes.append(plate(name, face, -length / 2, -length / 2 + width, z1, z2, 0, depth)[:2])
            meshes.append(plate(name, face, length / 2 - width, length / 2, z1, z2, 0, depth)[:2])
        elif row["forma"] == "arco":                       # Rectangle up to z2, half circle on top
            outline = [(-half, z1), (half, z1)]
            for k in range(ARCH_SEGMENTS + 1):
                angle = math.pi * k / ARCH_SEGMENTS
                outline.append((half * math.cos(angle), z2 + half * math.sin(angle)))
            outline.append((-half, z1))
            outline = outline[:-1]
            meshes.append(plate(name, face, 0, 0, z1, z2, 0, depth, outline)[:2])
        elif row["forma"] == "reja":                        # Vertical bars, a top and a bottom bar, in front
            bar = 0.04
            count = int(width / BAR_SPACING)
            for k in range(count + 1):
                s = -half + k * width / count
                meshes.append(plate(name, face, s - bar / 2, s + bar / 2, z1, z2, depth - bar, depth)[:2])
            for z in (z1, (z1 + z2) / 2, z2 - bar):
                meshes.append(plate(name, face, -half, half, z, z + bar, depth - bar, depth)[:2])
    return meshes

In [ ]:
# Celda 5a
# Roof models. Each one returns (error, parameters) for the building points inside a polygon (shrunk 0.4 m).
# The cloud is read in chunks, keeping only x, y, z of the building points (class 6), to save memory:
chunks = []
with laspy.open(CLOUD) as reader:
    for points in reader.chunk_iterator(1_000_000):
        building = np.array(points.classification) == 6
        chunks.append(np.c_[np.array(points.x)[building] - ORIGIN[0], np.array(points.y)[building] - ORIGIN[1], np.array(points.z)[building]])
bx, by, bz = np.vstack(chunks).T
del chunks
sample_rng = np.random.default_rng(1)

def robust_fit(A, z):
    """Least squares that leaves out outliers while fitting, but is scored on all the points: mean absolute
    residual, each one truncated at 2 m (a chapel 5 m lower than the model counts, but does not dominate)."""
    use = np.ones(len(z), bool)
    for _ in range(3):
        coef = np.linalg.lstsq(A[use], z[use], rcond=None)[0]
        res = z - A @ coef
        mad = np.median(np.abs(res[use] - np.median(res[use]))) + 1e-3
        use = np.abs(res) < max(3 * 1.4826 * mad, 0.3)
    return coef, float(np.mean(np.minimum(np.abs(res), 2.0)))

def axes(polygon):
    """Center and the two side directions (long side first) of the smallest rectangle around a polygon."""
    rect = polygon.minimum_rotated_rectangle
    c = list(rect.exterior.coords)
    sides = sorted([np.subtract(c[1], c[0]), np.subtract(c[2], c[1])], key=lambda e: -np.linalg.norm(e))
    return (rect.centroid.x, rect.centroid.y), [(e / np.linalg.norm(e), np.linalg.norm(e)) for e in sides]

def fit_roofs(polygon):
    inner = polygon.buffer(-0.4)
    if inner.is_empty:
        inner = polygon
    x0, y0, x1, y1 = inner.bounds
    sel = (bx > x0) & (bx < x1) & (by > y0) & (by < y1)
    sel[sel] = shapely.contains_xy(inner, bx[sel], by[sel])
    x, y, z = bx[sel], by[sel], bz[sel]
    if len(z) > ROOF_MAX_POINTS:
        k = sample_rng.choice(len(z), ROOF_MAX_POINTS, replace=False)
        x, y, z = x[k], y[k], z[k]
    if len(z) < 50:
        return None
    one = np.ones(len(z))
    models = {}
    coef, error = robust_fit(one[:, None], z)
    models["plana"] = (error, {"z": coef[0]})
    coef, error = robust_fit(np.c_[one, x, y], z)
    models["un_agua"] = (error, {"a": coef[0], "b": coef[1], "c": coef[2]})
    center, sides = axes(polygon)
    # Two pitches: the ridge runs along one side of the rectangle; u = meters across the ridge.
    for name, (along, _), (_, across) in (("dos_aguas_largo", sides[0], sides[1]), ("dos_aguas_corto", sides[1], sides[0])):
        normal = np.array([-along[1], along[0]])
        u = (x - center[0]) * normal[0] + (y - center[1]) * normal[1]
        best = None
        for u0 in np.arange(-across / 2 + 1, across / 2 - 1 + 1e-9, ROOF_STEP):
            A = np.c_[one, -np.maximum(u - u0, 0), -np.maximum(u0 - u, 0)]
            coef, error = robust_fit(A, z)
            if coef[1] <= 0.05 or coef[2] <= 0.05:   # Both pitches must fall from the ridge
                continue
            if best is None or error < best[0]:
                best = (error, {"center": center, "normal": normal, "u0": u0, "ridge": coef[0], "k1": coef[1], "k2": coef[2]})
        if best:
            models[name] = best
    return models, len(z)

def best_roof(polygon):
    fitted = fit_roofs(polygon)
    if fitted is None:
        return None
    models, count = fitted
    lowest = min(e for e, _ in models.values())
    for name in ["plana", "un_agua", "dos_aguas_largo", "dos_aguas_corto"]:   # Simplest first
        if name in models and models[name][0] <= lowest + ROOF_SIMPLER:
            return {"model": name, "error": models[name][0], "params": models[name][1], "points": count}

def roof_z(roof, x, y):
    p = roof["params"]
    if roof["model"] == "plana":
        return p["z"]
    if roof["model"] == "un_agua":
        return p["a"] + p["b"] * x + p["c"] * y
    u = (x - p["center"][0]) * p["normal"][0] + (y - p["center"][1]) * p["normal"][1]
    return p["ridge"] - p["k1"] * max(u - p["u0"], 0) - p["k2"] * max(p["u0"] - u, 0)

In [ ]:
# Celda 5b
# Cut a part into pieces until each one is explained by a simple roof.
def cut_in_two(polygon, direction, center, position):
    point = np.array(center) + direction * position
    normal = np.array([-direction[1], direction[0]])
    line = LineString([tuple(point - normal * 1000), tuple(point + normal * 1000)])
    pieces = [g for g in split(polygon, line).geoms if g.area > 0]
    return pieces if len(pieces) == 2 else None

def roof_pieces(polygon, depth=0):
    roof = best_roof(polygon)
    if roof is None:
        return [(polygon, None)]
    if roof["error"] <= ROOF_OK or depth >= ROOF_MAX_DEPTH or polygon.area < 2 * ROOF_MIN_AREA:
        return [(polygon, roof)]
    center, sides = axes(polygon)
    best = None
    for direction, length in sides:
        for fraction in np.arange(0.1, 0.91, 0.05):
            pieces = cut_in_two(polygon, direction, center, (fraction - 0.5) * length)
            if pieces is None or min(p.area for p in pieces) < ROOF_MIN_AREA:
                continue
            roofs = [best_roof(p) for p in pieces]
            if any(r is None for r in roofs):
                continue
            error = sum(r["error"] * r["points"] for r in roofs) / sum(r["points"] for r in roofs)
            if best is None or error < best[0]:
                best = (error, pieces)
    if best is None or best[0] > roof["error"] * ROOF_CUT_GAIN:
        return [(polygon, roof)]
    return [piece for p in best[1] for piece in roof_pieces(p, depth + 1)]

# Mesh of one piece: the roof (split along the ridge, so that every triangle lies on one pitch) and the walls on
# its border, from z_base up to the roof. With eaves, the roof covers the larger polygon `eaves` (same planes).
def roof_piece_mesh(polygon, roof, z_base, eaves=None):
    polygon = orient(polygon, 1.0)
    def halves_of(shape):
        if not roof["model"].startswith("dos_aguas"):
            return [shape]
        p = roof["params"]
        along = np.array([p["normal"][1], -p["normal"][0]])
        point = np.array(p["center"]) + p["normal"] * p["u0"]
        ridge = LineString([tuple(point - along * 1000), tuple(point + along * 1000)])
        return [orient(g, 1.0) for g in split(shape, ridge).geoms if g.area > 0]
    vertices, faces = [], []
    z0 = ORIGIN[2]
    for half in halves_of(orient(eaves, 1.0) if eaves is not None else polygon):
        for triangle in shapely.constrained_delaunay_triangles(half).geoms:
            triangle = orient(triangle, 1.0)
            start = len(vertices)
            for x, y in list(triangle.exterior.coords)[:3]:
                vertices.append((x, y, roof_z(roof, x, y) - z0))
            faces.append((start, start + 1, start + 2))
    for half in halves_of(polygon):
        for ring_line in [half.exterior] + list(half.interiors):
            coords = list(ring_line.coords)
            for (ax, ay), (cx, cy) in zip(coords[:-1], coords[1:]):
                if polygon.boundary.distance(Point((ax + cx) / 2, (ay + cy) / 2)) > 1e-6:
                    continue   # The cut along the ridge is not a wall
                start = len(vertices)
                vertices += [(ax, ay, z_base - z0), (cx, cy, z_base - z0), (cx, cy, roof_z(roof, cx, cy) - z0), (ax, ay, roof_z(roof, ax, ay) - z0)]
                faces.append((start, start + 1, start + 2, start + 3))
    return vertices, faces

In [ ]:
# Celda 5c
# Replace the LOD2 grid of each part of the ensemble by its pieces (unless the error stays above ROOF_KEEP):
with open(IN_JSON, encoding="utf-8") as f:
    meshes = json.load(f)["parts"]
footprints_catastro = gpd.read_file(GPKG, layer="building_parts")
block_union = shapely.union_all([shapely.affinity.translate(g, -ORIGIN[0], -ORIGIN[1]) for g in footprints_catastro.geometry]).buffer(0.01)
by_name = {m["name"]: m for m in meshes}
report = []
replaced = 0
for _, row in footprints_catastro.iterrows():
    name = str(row["localId"])
    mesh = by_name.get(name)
    if mesh is None or mesh.get("lod") != 2 or name in KEEP_GRID:
        continue
    polygon = shapely.affinity.translate(row.geometry, -ORIGIN[0], -ORIGIN[1])
    if polygon.geom_type == "MultiPolygon":
        polygon = max(polygon.geoms, key=lambda g: g.area)
    if name in MANUAL_BAYS:
        bays = []
        rest = polygon
        for cx, cy, length, width, angle in MANUAL_BAYS[name]:
            a = math.radians(angle)
            u, n = np.array([math.cos(a), math.sin(a)]), np.array([-math.sin(a), math.cos(a)])
            rectangle = shapely.Polygon([tuple(np.array([cx, cy]) + i * length / 2 * u + j * width / 2 * n) for i, j in ((-1, -1), (1, -1), (1, 1), (-1, 1))])
            bay = rest.intersection(rectangle)
            for component in getattr(bay, "geoms", [bay]):
                if component.geom_type == "Polygon" and component.area > 1:
                    bays += roof_pieces(component)   # A drawn bay is also cut if one roof does not explain it
            rest = rest.difference(rectangle)
        for piece in getattr(rest, "geoms", [rest]):
            if piece.geom_type == "Polygon" and piece.area > 5:
                bays += roof_pieces(piece)
    else:
        bays = roof_pieces(polygon)
    fitted = [(p, r) for p, r in bays if r]
    points = sum(r["points"] for _, r in fitted)
    error = sum(r["error"] * r["points"] for _, r in fitted) / points if points else float("inf")
    keep = (error > ROOF_KEEP and name not in MANUAL_BAYS and name not in FORCE_BAYS) or len(fitted) < len(bays)
    for k, (piece, roof) in enumerate(bays):
        report.append({"parte": name, "pieza": k, "modelo": roof["model"] if roof else "", "error_m": round(roof["error"], 3) if roof else "",
                       "puntos": roof["points"] if roof else 0, "area_m2": round(piece.area, 1), "usada": not keep})
    print(name, len(bays), "pieces, error", round(error, 2), "m", "-> keeps the grid" if keep else "")
    if keep:
        continue
    z_base = min(z for x, y, z in mesh["vertices"]) + ORIGIN[2]
    new = []
    for k, (piece, roof) in enumerate(bays):
        eaves = None
        if mesh["parcel"] in EAVE_PARCELS:   # The tiles run out over the cornice, toward the street or the courtyard only
            grown = piece.union(piece.buffer(EAVE, join_style="mitre").difference(block_union))
            eaves = max(getattr(grown, "geoms", [grown]), key=lambda g: g.area)
        vertices, faces = roof_piece_mesh(piece, roof, z_base, eaves)
        new.append({**{key: mesh[key] for key in ("parcel", "group", "height_m")}, "name": f"{name}_c{k}", "lod": 2,
                    "height_method": f"crujía: {roof['model']}, error medio {roof['error']:.2f} m", "vertices": vertices, "faces": faces})
    meshes = [m for m in meshes if m["name"] != name] + new
    replaced += 1
with open(ROOF_REPORT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(report[0]))
    writer.writeheader()
    writer.writerows(report)
print(replaced, "parts replaced by bays; report in", ROOF_REPORT)

In [ ]:
# Celda 5d
# Estimated windows on the walls without photos. Outer walls of the bays (to a street or a courtyard: on the border
# of the union of all the footprints of the block, so not party walls) are grouped by line (same parcel, direction
# and offset), so that a wall cut in short pieces counts as one. Windows take the measured levels and rhythm of the
# Toledo facade: axes every FILL_AXIS m centered on the whole wall, one row per level that fits under the eaves
# (fitted to the cloud). On walls drawn from a photo, only above everything measured; on walls with estimated
# elements, only above the openings within FILL_NEAR_OPENING m. All [ESTIMADO].
FILL_PARCELS = ["0139424VK4703G", "0139423VK4703G", "0139402VK4703G"]
FILL_ROWS = [(647.0, 649.0), (650.6, 652.5), (655.1, 656.7), (657.95, 659.45)]   # Toledo 2.6-4.6 ... 13.55-15.05 m + 644.4
FILL_AXIS = 3.9          # Distance between window axes on Toledo (m)
FILL_WIDTH = 1.3         # Opening; the stone frame is FILL_FRAME wider on each side
FILL_FRAME = 0.25
FILL_MIN_WALL = 4.0      # Shorter walls (whole line) get no windows
FILL_EAVE_GAP = 0.8      # Top of a window at least this far under the eaves
FILL_GROUND_GAP = 0.6    # Bottom of a window at least this far above the street / courtyard (terrain of 13)
FILL_ABOVE_MEASURED = 0.8   # Gap between the top of the measured elements of a wall and the first estimated window
FILL_SKIP = ["remate", "cruz", "aguja"]   # LOD3 pieces that are not on a wall

all_footprints = shapely.union_all([shapely.affinity.translate(g, -ORIGIN[0], -ORIGIN[1]) for g in footprints_catastro.geometry])
outline = all_footprints.boundary
measured = [m for m in meshes if m.get("lod") == 3 and not any(k in m["name"] for k in FILL_SKIP)]
measured_points = np.array([np.mean(m["vertices"], axis=0) for m in measured]) if measured else np.zeros((0, 3))
measured_tops = np.array([max(z for x, y, z in m["vertices"]) for m in measured]) if measured else np.zeros(0)
# Measured on a photo ("medido") or estimated; and openings (glass, dark arches) versus bands and plates:
measured_photo = np.array([str(m.get("height_method", "")).startswith("medido") for m in measured], bool)
measured_opening = np.array([m.get("group") == "vidrio" or m.get("material_hint") in ("ventana", "sombra") for m in measured], bool)
FILL_NEAR_OPENING = 2.5     # On walls without photo: an estimated opening within this distance (m) blocks a window

with open(TERRAIN_JSON, encoding="utf-8") as f:
    terrain_mesh = [m for m in json.load(f)["parts"] if m["name"] == "terreno"][0]
terrain_xy = np.array(terrain_mesh["vertices"])[:, :2]
terrain_z = np.array(terrain_mesh["vertices"])[:, 2]

def ground_at(x, y):
    """Lowest terrain point within 3 m of (x, y), local z."""
    d2 = (terrain_xy[:, 0] - x) ** 2 + (terrain_xy[:, 1] - y) ** 2
    return float(terrain_z[d2 < 9.0].min()) if (d2 < 9.0).any() else float(terrain_z.min())

# Outer wall quads, grouped by line:
candidates = []
for mesh in [m for m in meshes if m.get("lod") == 2 and m["parcel"] in FILL_PARCELS]:
    for face in mesh["faces"]:
        if len(face) != 4:
            continue
        (ax, ay, a_low), (cx, cy, c_low), (_, _, c_top), (_, _, a_top) = [mesh["vertices"][i] for i in face]
        length = math.hypot(cx - ax, cy - ay)
        if length < 0.3 or abs(a_low - c_low) > 0.01:
            continue                                     # Not a wall quad from the base up
        middle = Point((ax + cx) / 2, (ay + cy) / 2)
        if outline.distance(middle) > 0.3:
            continue                                     # Party wall or wall inside the block
        t = np.array([cx - ax, cy - ay]) / length
        if t[0] < 0 or (t[0] == 0 and t[1] < 0):         # Same direction for both ways of drawing the line
            t = -t
        n = np.array([t[1], -t[0]])
        candidates.append({"parcel": mesh["parcel"], "angle": math.degrees(math.atan2(t[1], t[0])), "t": t, "n": n,
                           "offset": float(np.dot([ax, ay], n)), "quad": (ax, ay, a_top, cx, cy, c_top)})

# Walls cut in pieces with slightly different angles: pieces join a wall when they differ by less than
# FILL_JOIN_ANGLE degrees and FILL_JOIN_OFFSET m (single linkage), so the rhythm runs along the whole wall.
FILL_JOIN_ANGLE = 6.0
FILL_JOIN_OFFSET = 0.6
parent = list(range(len(candidates)))
def root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for i, a in enumerate(candidates):
    for j in range(i + 1, len(candidates)):
        b = candidates[j]
        if a["parcel"] == b["parcel"] and abs(a["angle"] - b["angle"]) < FILL_JOIN_ANGLE                 and abs(float(np.dot(b["quad"][:2], a["n"])) - a["offset"]) < FILL_JOIN_OFFSET:
            parent[root(j)] = root(i)
groups = {}
for i, a in enumerate(candidates):
    g = groups.setdefault(root(i), {"t": candidates[root(i)]["t"], "n": candidates[root(i)]["n"], "parcel": a["parcel"], "quads": []})
    g["quads"].append(a["quad"])

filled = []
for key, g in groups.items():
    t, n = g["t"], g["n"]
    origin = np.array(g["quads"][0][:2])
    quads = []
    for ax, ay, a_top, cx, cy, c_top in g["quads"]:
        sa, sc = float(np.dot([ax, ay] - origin, t)), float(np.dot([cx, cy] - origin, t))
        a_xy, c_xy = np.array([ax, ay]), np.array([cx, cy])
        if sa > sc:
            sa, sc, a_top, c_top, a_xy, c_xy = sc, sa, c_top, a_top, c_xy, a_xy
        quads.append((sa, sc, a_top, c_top, a_xy, c_xy))
    s_min, s_max = min(q[0] for q in quads), max(q[1] for q in quads)
    length = s_max - s_min
    if length < FILL_MIN_WALL:
        continue
    offset = float(np.mean([np.dot(q[:2], n) for q in g["quads"]]))
    middle = origin + t * (s_min + s_max) / 2
    normal = n if not all_footprints.contains(Point(*(middle + n * 0.5))) else -n   # Outward
    rel = measured_points[:, :2] - origin
    along = rel @ t
    on_wall = (np.abs(measured_points[:, :2] @ n - offset) < 1.5) & (along > s_min) & (along < s_max)
    # Wall drawn from a photo: nothing is added under what was measured. Otherwise only nearby openings block.
    photo_wall = (on_wall & measured_photo).any()
    wall_top = measured_tops[on_wall].max() + ORIGIN[2] if on_wall.any() else -np.inf
    count = int((length - 1.0) // FILL_AXIS)
    for k in range(count):
        s = s_min + (length - (count - 1) * FILL_AXIS) / 2 + k * FILL_AXIS
        holder = [q for q in quads if q[0] <= s <= q[1]]
        if not holder or s - FILL_WIDTH / 2 < s_min or s + FILL_WIDTH / 2 > s_max:
            continue                                     # Off the wall
        sa, sc, a_top, c_top, a_xy, c_xy = holder[0]
        eave = a_top + (c_top - a_top) * (s - sa) / (sc - sa) + ORIGIN[2]
        point = a_xy + (c_xy - a_xy) * (s - sa) / (sc - sa)   # On the quad itself (a wall line can step up to 0.6 m)
        if photo_wall:
            measured_top = wall_top
        else:
            near = on_wall & measured_opening & (np.abs(along - s) < FILL_NEAR_OPENING)
            measured_top = measured_tops[near].max() + ORIGIN[2] if near.any() else -np.inf
        ground = ground_at(*point) + ORIGIN[2]
        for z1, z2 in FILL_ROWS:
            if z2 + FILL_EAVE_GAP > eave or z1 - FILL_GROUND_GAP < ground or z1 - FILL_ABOVE_MEASURED < measured_top:
                continue
            for kind, half, n1, n2, zz1, zz2, material in (
                    ("marco", FILL_WIDTH / 2 + FILL_FRAME, 0.0, 0.06, z1 - 0.15, z2 + 0.15, "granito_liso"),
                    ("hueco", FILL_WIDTH / 2, 0.06, 0.07, z1, z2, "ventana")):
                vertices = []
                for along_s in (s - half, s + half):
                    for nn in (n1, n2):
                        for z in (zz1, zz2):
                            p = point + t * (along_s - s) + normal * nn
                            vertices.append((float(p[0]), float(p[1]), z - ORIGIN[2]))
                faces = [(0, 1, 3, 2), (4, 6, 7, 5), (0, 4, 5, 1), (2, 3, 7, 6), (0, 2, 6, 4), (1, 5, 7, 3)]
                filled.append({"name": f"hueco_estimado_{len(filled)}_{kind}", "parcel": g["parcel"], "group": "volumen",
                               "material_hint": material, "lod": 3, "height_m": 0.0,
                               "height_method": "[ESTIMADO] niveles y ritmo de la fachada a Toledo (sin foto de este muro)",
                               "vertices": vertices, "faces": faces})
meshes += filled
print(len(filled) // 2, "estimated windows on", len(groups), "outer walls without photos")

In [ ]:
# Celda 5
# Start from the LOD3: drop the replaced elements and cut the LOD2 parts of the towers and the dome flat.
kept = [m for m in meshes if not any(m["name"].startswith(prefix) for prefix in DROP_ELEMENTS)]
print(len(meshes) - len(kept), "LOD3 elements dropped")

# Footprint of each tower and of the dome (first ring of its widest piece, plus CLIP_MARGIN): every LOD2 vertex
# of the ensemble inside it goes down to the cut height, whatever part it belongs to.
footprints = {}
for name in CLIP_Z:
    own = [p for p in pieces if frame_name(p) == name]
    widest = max(own, key=lambda p: float(pieces[p][0]["hu"]) * float(pieces[p][0]["hv"]))
    row = pieces[widest][0]
    corners = [to_local(name, a, b, 0)[:2] for a, b in ring(float(row["hu"]), float(row["hv"]), row["chaflan"])]
    footprints[name] = shapely.Polygon(corners).buffer(CLIP_MARGIN)

for mesh in kept:
    # Only grid parts are cut. Bays are never cut: a roof pitch is a few large triangles, and moving one corner
    # tilts the whole pitch (the nave dropped up to 2 m this way; found by 16_control).
    if mesh.get("lod") != 2 or "_c" in mesh["name"].split("_part")[-1]:
        continue
    points = np.array(mesh["vertices"])
    changed = 0
    for name, footprint in footprints.items():
        clip = CLIP_Z[name] - ORIGIN[2]
        cut = shapely.contains_xy(footprint, points[:, 0], points[:, 1]) & (points[:, 2] > clip)
        points[cut, 2] = clip
        changed += int(cut.sum())
    if mesh["name"] in KEEP_GRID:   # Towers and dome parts: a flat prism under the new pieces
        base = points[:, 2].min()
        top = min(CLIP_Z[name] for name in CLIP_Z if shapely.contains_xy(footprints[name], points[:, 0].mean(), points[:, 1].mean())) - ORIGIN[2] if any(shapely.contains_xy(footprints[name], points[:, 0].mean(), points[:, 1].mean()) for name in CLIP_Z) else points[:, 2].max()
        points[points[:, 2] > base + 0.01, 2] = top
    mesh["vertices"] = points.tolist()
    if changed:
        print(mesh["name"], changed, "vertices cut under the towers / dome")

In [ ]:
# Celda 6
# New pieces, openings and tower spires:
def new_part(name, material, vertices, faces, origin_text):
    return {"name": name, "parcel": "0139402VK4703G", "group": "volumen", "material_hint": material, "lod": 3,
            "height_m": 0.0, "height_method": origin_text, "vertices": vertices, "faces": faces}

added = []
for name, rows in pieces.items():
    vertices, faces = piece_mesh(name, rows)
    added.append(new_part(name, rows[0]["material"], vertices, faces, rows[0]["origen"]))
for row in openings:
    for k, (vertices, faces) in enumerate(opening_meshes(row)):
        added.append(new_part(f"{row['pieza']}_{row['forma']}_{k}", row["material"], vertices, faces, row["origen"]))

def box(name, frame, a1, a2, b1, b2, z1, z2):
    corners = [(a1, b1), (a2, b1), (a2, b2), (a1, b2)]
    vertices = [to_local(frame, a, b, z1) for a, b in corners] + [to_local(frame, a, b, z2) for a, b in corners]
    faces = [(3, 2, 1, 0), (4, 5, 6, 7), (0, 1, 5, 4), (1, 2, 6, 5), (2, 3, 7, 6), (3, 0, 4, 7)]
    return vertices, faces

for name, tip in TOWER_SPIRES.items():
    base = float(pieces[name + "_chapitel"][-1]["z"]) - 0.3
    vertices = [to_local(name, SPIRE_RADIUS * math.cos(2 * math.pi * k / 8), SPIRE_RADIUS * math.sin(2 * math.pi * k / 8), base)
                for k in range(8)] + [to_local(name, 0, 0, tip)]
    faces = [(k, (k + 1) % 8, 8) for k in range(8)] + [tuple(reversed(range(8)))]
    added.append(new_part(name + "_aguja", "pizarra", vertices, faces, "punta medida en la nube; radio [ESTIMADO]"))
    half = CROSS_BAR / 2
    added.append(new_part(name + "_cruz_mastil", "hierro", *box(name + "_cruz_mastil", name, -half, half, -half, half, tip - 0.2, tip + CROSS_HEIGHT), "[ESTIMADO]"))
    arm_z = tip + CROSS_HEIGHT * 0.65
    added.append(new_part(name + "_cruz_brazo", "hierro", *box(name + "_cruz_brazo", name, -CROSS_ARM / 2, CROSS_ARM / 2, -half, half, arm_z - half, arm_z + half), "[ESTIMADO]"))

result = kept + added
print(len(added), "new meshes |", len(result), "in total")

In [ ]:
# Celda 7
# Side chapels of the Colegiata, measured on the cloud along both sides of the nave and shaped after the aerial views
# (source 14, reference only): buttresses with a curved back ("aletones") between the chapels, with a tile coping;
# a skylight box between two of them; one window per bay on the nave wall above the chapel roofs. Also the west gable
# of the nave (window, oculus, moulding, raking cornice), measured on an orthographic elevation of the mesh of the
# Ayuntamiento. All in the frame "nave". Shapes: "aleton" (prism of the outline under the back, with a tile coping),
# "perfil" (prism of a closed outline v:z, counterclockwise), "perfil_v" (outline u:z on a wall facing -v),
# "caja" and "ventana" (boxes). The south gable of the transept (window, oval oculus) is in the same table.
chapel_rows = []
for table in (CHAPELS_CSV, GABLE_CSV):
    with open(table, encoding="utf-8") as f:
        chapel_rows += list(csv.DictReader(f))
COPING = 0.12   # Thickness of the tile coping on the back of a buttress, and its overhang on each side / 2 [ESTIMADO]

def prism_along_u(frame, u1, u2, outline):
    """Prism between u1 and u2 of an outline in (v, z), counterclockwise with v to the right and z up."""
    m = len(outline)
    vertices = [to_local(frame, u1, v, z) for v, z in outline] + [to_local(frame, u2, v, z) for v, z in outline]
    faces = [tuple(reversed(range(m))), tuple(range(m, 2 * m))] + [(k, (k + 1) % m, m + (k + 1) % m, m + k) for k in range(m)]
    return vertices, faces

def prism_along_v(frame, v1, v2, outline):
    """Prism between v1 and v2 of an outline in (u, z), counterclockwise with u to the right and z up (as seen
    from -v, the side a wall facing -v is looked at from)."""
    return prism_along_u_swapped(frame, v1, v2, outline[::-1])

def prism_along_u_swapped(frame, v1, v2, outline):
    m = len(outline)
    vertices = [to_local(frame, u, v1, z) for u, z in outline] + [to_local(frame, u, v2, z) for u, z in outline]
    faces = [tuple(reversed(range(m))), tuple(range(m, 2 * m))] + [(k, (k + 1) % m, m + (k + 1) % m, m + k) for k in range(m)]
    return vertices, faces

chapel_parts = []
for row in chapel_rows:
    if row["forma"] == "perfil_v":   # Outline u:z on a wall facing -v (gables of the transept)
        outline = [tuple(map(float, p.split(":"))) for p in row["perfil"].split(";")]
        chapel_parts.append(new_part(row["pieza"], row["material"],
                                     *prism_along_v("nave", float(row["v1"]), float(row["v2"]), outline), row["origen"]))
        continue
    u1, u2 = float(row["u1"]), float(row["u2"])
    if row["forma"] == "aleton":
        back = sorted((float(v), float(z)) for v, z in (p.split(":") for p in row["perfil"].split(";")))
        base = float(row["z1"])
        outline = [(back[0][0], base), (back[-1][0], base)] + back[::-1]
        chapel_parts.append(new_part(row["pieza"], row["material"], *prism_along_u("nave", u1, u2, outline), row["origen"]))
        coping = back + [(v, z + COPING) for v, z in back[::-1]]
        chapel_parts.append(new_part(row["pieza"] + "_albardilla", "teja",
                                     *prism_along_u("nave", u1 - COPING / 2, u2 + COPING / 2, coping), row["origen"]))
    elif row["forma"] == "perfil":
        outline = [tuple(map(float, p.split(":"))) for p in row["perfil"].split(";")]
        chapel_parts.append(new_part(row["pieza"], row["material"], *prism_along_u("nave", u1, u2, outline), row["origen"]))
    else:   # "caja" and "ventana": a box
        v1, v2, z1, z2 = (float(row[k]) for k in ("v1", "v2", "z1", "z2"))
        chapel_parts.append(new_part(row["pieza"], row["material"], *box(row["pieza"], "nave", u1, u2, v1, v2, z1, z2), row["origen"]))
print(len(chapel_parts), "meshes of the chapels")

result = result + chapel_parts
with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump({"origin": ORIGIN, "parts": result}, f)
print(len(result), "meshes saved")